# Memory Loop Deep Dive

Deep dive into every query the CLI app issues — each cell imports from the `memory/` package, so the code below is the code `app.py` runs. See the project README for context.

> **Note:** This is a copy of the notebook that ships inside the demo app at
> `apps/rag-to-memory-systems-demo/notebooks/`. It imports that app's `memory/`
> and `data/` packages directly; the first cell locates the app directory
> automatically, wherever you launch the kernel from inside the repo.

## Prerequisites

Run the one-time setup commands below from the app directory
(`apps/rag-to-memory-systems-demo/`), since they use paths relative to it:

- Oracle AI Database Free running at `localhost:1521/FREEPDB1`
- `DB_DEVELOPER_ROLE` and `CTXAPP` granted to the user in `.env`
- ONNX model loaded (`python -m memory.onnx_loader`)
- Demo tenant seeded (`python -m data.seed`)

## 1. Environment & connection

In [ ]:
import os, json, asyncio, sys
from pathlib import Path

# This notebook is a copy of the one that ships inside the demo app at
# apps/rag-to-memory-systems-demo/notebooks/. The demo code lives in that app's
# `memory/` and `data/` packages, so locate the app root and make it importable
# — wherever the kernel's cwd happens to be inside the repo.
_APP_REL = Path("apps/rag-to-memory-systems-demo")

def _find_app_root() -> Path:
    here = Path(os.getcwd()).resolve()
    for base in [here, *here.parents]:
        # Repo checkout: app lives under apps/rag-to-memory-systems-demo/.
        if (base / _APP_REL / "memory").is_dir():
            return base / _APP_REL
        # Already inside the app itself (cwd is the app root or its notebooks/).
        if (base / "memory").is_dir() and (base / "data").is_dir():
            return base
    raise RuntimeError(
        f"Could not find {_APP_REL}/memory starting from {here}. "
        "Launch this notebook from inside the oracle-ai-developer-hub repo."
    )

APP_ROOT = _find_app_root()
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))
print("App root:", APP_ROOT)

from dotenv import load_dotenv
# The app's .env is not an ancestor of this notebook, so load it by absolute
# path. (memory.db also calls load_dotenv() at import; loading here first means
# os.environ is already populated before any connection is opened.)
load_dotenv(APP_ROOT / ".env")

from memory.db import connect_sync, connect
from memory.startup import lexical_available

conn = connect_sync()
print("Oracle version:", conn.version)

async def check():
    aconn = await connect()
    lex = await lexical_available(aconn)
    await aconn.close()
    return lex

print("Oracle Text available (CTXAPP):", await check())

## 2. The five schemas

Five typed tables plus a `deletion_events` audit table for the GDPR
cascade at the end. Notable constraints created with them:

- `idx_fact_text` and `idx_ep_text` are Oracle Text CONTEXT indexes
  on `fact_memory.content` and `episodic_memory.summary`. They power
  the lexical half of hybrid retrieval.
- `ck_trace_event_type` is a CHECK constraint on `trace_memory.event_type`,
  pinning the enum to the eight values the manager writes
  (`user_msg`, `model_msg`, `tool_call`, `tool_result`, `turn_envelope`,
  `confirmation_synthesis`, `extraction_error`, `promotion_error`).


In [2]:
from memory.ddl import create_all, drop_all, TABLES
drop_all(conn)
create_all(conn)
cur = conn.cursor()
cur.execute("SELECT table_name FROM user_tables WHERE LOWER(table_name) IN "
            + "(" + ",".join(f"'{t}'" for t in TABLES) + ")")
print("Tables created:", sorted(r[0].lower() for r in cur))

Tables created: ['deletion_events', 'episodic_memory', 'fact_memory', 'policy_memory', 'preference_memory', 'trace_memory']


## 3. Load the ONNX embedding model

Loaded once into the database via `DBMS_VECTOR.LOAD_ONNX_MODEL`. After that,
embeddings happen in SQL via `VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :text AS DATA)`
— no Python-side embedding step at query time.

In [3]:
from memory.embeddings import MODEL_NAME, VECTOR_DIM, vector_embedding_sql

# Smoke-test that the model is loaded:
cur.execute(f"SELECT {vector_embedding_sql(':t')} FROM dual", t="customer support")
(vec,) = cur.fetchone()
print(f"Model: {MODEL_NAME}, dimension: {len(vec)} (expected {VECTOR_DIM})")

Model: ALL_MINILM_L12_V2, dimension: 384 (expected 384)


## 4. Seed the demo tenant

Five policies, three preferences, eight facts (five customer-scoped + three tenant-wide), three prior episodes — including the canonical Stripe webhook signature-mismatch example.

In [4]:
import data.seed
await data.seed.seed()

Seeded tenant acme-support: 5 policies, 3 preferences, 8 facts, 3 episodes.


## 5. Verify what's in each table

A quick SELECT per table to ground the rest of the notebook.

In [5]:
import oracledb

def _read_clob(cur, val):
    """Sync helper: read CLOB if needed, else return as-is."""
    if val is None:
        return None
    if hasattr(val, "read"):
        return val.read()
    return val

for tbl, cols in [
    ("policy_memory", "policy_key, JSON_SERIALIZE(policy_value RETURNING VARCHAR2(200))"),
    ("preference_memory", "pref_key, JSON_SERIALIZE(pref_value RETURNING VARCHAR2(200))"),
    ("fact_memory", "subject, predicate, DBMS_LOB.SUBSTR(content, 80, 1), status"),
    ("episodic_memory", "task_type, title, outcome"),
]:
    print(f"\n-- {tbl} --")
    cur.execute(f"SELECT {cols} FROM {tbl}")
    for row in cur:
        print(" ", row)


-- policy_memory --
  ('refund_threshold', '{"max_auto_approve_usd":500}')
  ('tone_guardrail', '{"forbidden_phrases":["unfortunately, our policy"]}')
  ('data_residency', '{"allowed_regions":["us-east-1"]}')
  ('escalation_path', '{"trigger":"negative_sentiment","after_turns":3,"queue":"tier2"}')
  ('pii_redaction', '{"fields":["ssn","card_number","cvv"],"mode":"mask"}')

-- preference_memory --
  ('notification_channel', '{"channel":"email"}')
  ('timezone', '{"tz":"America/New_York"}')
  ('language', '{"locale":"en-US"}')

-- fact_memory --
  ('customer:jane_doe@example.com', 'integration', 'Uses Stripe Connect with platform model; charges flow through their platform acc', 'active')
  ('customer:jane_doe@example.com', 'stack', 'Primary stack: Python 3.11 + FastAPI on Kubernetes (EKS)', 'active')
  ('product:stripe', 'webhook_signing', 'Stripe webhooks are signed with HMAC-SHA256 over the raw request body; the signa', 'active')
  ('product:stripe', 'webhook_retries', 'Webhook retrie

## 6. Path A — known-scope lookup

Policy and preference retrieval. Exact-match SQL, exhaustive, no top-k cutoff,
no ranking. This is what feeds the static prefix of the prompt — the part
that should hit the cache and stay stable across turns.

In [6]:
import os
TENANT = os.getenv("DEMO_TENANT_ID", "acme-support")
USER = os.getenv("DEMO_USER_ID", "customer:jane_doe@example.com")

cur.execute("""
SELECT 'policy' AS kind, policy_key AS key,
       JSON_SERIALIZE(policy_value RETURNING VARCHAR2(200)) AS value
  FROM policy_memory
 WHERE tenant_id = :tid
   AND (effective_until IS NULL OR effective_until > SYSTIMESTAMP)
UNION ALL
SELECT 'preference', pref_key, JSON_SERIALIZE(pref_value RETURNING VARCHAR2(200))
  FROM preference_memory
 WHERE tenant_id = :tid AND user_id = :user_id
ORDER BY kind, key
""", tid=TENANT, user_id=USER)

for kind, key, val in cur:
    print(f"  {kind:<10} {key:<25} {val}")

  policy     data_residency            {"allowed_regions":["us-east-1"]}
  policy     escalation_path           {"trigger":"negative_sentiment","after_turns":3,"queue":"tier2"}
  policy     pii_redaction             {"fields":["ssn","card_number","cvv"],"mode":"mask"}
  policy     refund_threshold          {"max_auto_approve_usd":500}
  policy     tone_guardrail            {"forbidden_phrases":["unfortunately, our policy"]}
  preference language                  {"locale":"en-US"}
  preference notification_channel      {"channel":"email"}
  preference timezone                  {"tz":"America/New_York"}


## 7. Path B — hybrid retrieval, broken apart

Same fact-memory rows three different ways: vector-only, lexical-only, fused.
The fusion produces the score that the unified query uses for ranking.

In [7]:
from memory.embeddings import vector_embedding_sql
query = "stripe"

# Vector-only.
cur.execute(f"""
SELECT fact_id, subject, predicate,
       VECTOR_DISTANCE(embedding, {vector_embedding_sql(':q')}, COSINE) AS dist
  FROM fact_memory WHERE tenant_id = :tid AND status = 'active'
 ORDER BY dist FETCH FIRST 5 ROWS ONLY
""", tid=TENANT, q=query)
print("Vector-only:")
for r in cur:
    print(f"  dist={r[3]:.4f}  {r[1]}/{r[2]}")

Vector-only:
  dist=0.4933  customer:jane_doe@example.com/infrastructure
  dist=0.5134  product:stripe/webhook_signing
  dist=0.5384  customer:jane_doe@example.com/integration
  dist=0.7087  customer:jane_doe@example.com/account
  dist=0.7919  customer:jane_doe@example.com/deployment


In [8]:
# Lexical-only.
# Oracle Text treats a space-separated phrase as a strict sequence match.
# To match documents containing ANY of these tokens, we OR-join them — this
# matches what `memory.retrieval._sanitize_for_contains` does in production.
lex_query = " OR ".join(query.split())
print(f"Lexical CONTAINS query: {lex_query!r}\n")

cur.execute("""
SELECT fact_id, subject, predicate, SCORE(1) AS s
  FROM fact_memory
 WHERE tenant_id = :tid AND status = 'active'
   AND CONTAINS(content, :q, 1) > 0
 ORDER BY s DESC FETCH FIRST 5 ROWS ONLY
""", tid=TENANT, q=lex_query)
print("Lexical-only:")
for r in cur:
    print(f"  score={r[3]:>5}  {r[1]}/{r[2]}")

Lexical CONTAINS query: 'stripe'

Lexical-only:
  score=    9  product:stripe/webhook_signing
  score=    9  customer:jane_doe@example.com/infrastructure
  score=    4  customer:jane_doe@example.com/integration


In [9]:
# Fused (0.4*vector + 0.6*lexical fusion).
# Vector side uses the raw query (semantic similarity).
# Lexical side uses OR-joined tokens (any-term match).
lex_query = " OR ".join(query.split())

cur.execute(f"""
WITH
v AS (SELECT fact_id, DBMS_LOB.SUBSTR(content, 200, 1) AS content_preview,
              VECTOR_DISTANCE(embedding, {vector_embedding_sql(':q')}, COSINE) d
        FROM fact_memory WHERE tenant_id = :tid AND status = 'active'
       ORDER BY d FETCH FIRST 20 ROWS ONLY),
l AS (SELECT fact_id, DBMS_LOB.SUBSTR(content, 200, 1) AS content_preview, SCORE(1) s
        FROM fact_memory
       WHERE tenant_id = :tid AND status = 'active'
         AND CONTAINS(content, :lex, 1) > 0
       ORDER BY s DESC FETCH FIRST 20 ROWS ONLY),
m AS (SELECT NULLIF(MAX(s), 0) max_l FROM l)
SELECT COALESCE(v.fact_id, l.fact_id),
       CASE WHEN v.d IS NOT NULL AND l.s IS NOT NULL THEN
              0.4 * (1.0/(1.0+v.d)) + 0.6 * (l.s/m.max_l)
            WHEN v.d IS NOT NULL THEN 1.0/(1.0+v.d)
            ELSE l.s/m.max_l END AS score
  FROM v FULL OUTER JOIN l ON v.fact_id = l.fact_id CROSS JOIN m
 ORDER BY score DESC FETCH FIRST 5 ROWS ONLY
""", tid=TENANT, q=query, lex=lex_query)
print("Fused:")
for r in cur:
    print(f"  score={r[1]:.4f}  {r[0]}")

Fused:
  score=0.8679  fact_b4fc6396de81
  score=0.8643  fact_628542bffe38
  score=0.5853  fact_22cf8605502b
  score=0.5581  fact_0c8b7023422d
  score=0.5267  fact_64549c668d64


## 8. The unified CTE

The same hybrid logic, plus Path A enumeration for policy and preference,
plus the last 5 trace events. ONE round trip returns rows of every kind.

In [10]:
from memory.retrieval import assemble

async def run_unified():
    aconn = await connect()
    try:
        result = await assemble(aconn, TENANT, USER, "run_demo_notebook", "stripe webhook")
        return result
    finally:
        await aconn.close()

result = await run_unified()
for row in result.rows:
    score = f"{row.rank_score:.3f}" if row.rank_score is not None else "  —  "
    print(f"  {row.kind:<10} score={score}  {json.dumps(row.payload)[:80]}")
print(f"\nTotals by kind: {result.counts()}")

  policy     score=  —    {"policy_key": "escalation_path", "policy_value": {"trigger": "negative_sentimen
  policy     score=  —    {"policy_key": "data_residency", "policy_value": {"allowed_regions": ["us-east-1
  policy     score=  —    {"policy_key": "pii_redaction", "policy_value": {"fields": ["ssn", "card_number"
  policy     score=  —    {"policy_key": "refund_threshold", "policy_value": {"max_auto_approve_usd": 500}
  policy     score=  —    {"policy_key": "tone_guardrail", "policy_value": {"forbidden_phrases": ["unfortu
  preference score=  —    {"pref_key": "timezone", "pref_value": {"tz": "America/New_York"}, "source": "on
  preference score=  —    {"pref_key": "notification_channel", "pref_value": {"channel": "email"}, "source
  preference score=  —    {"pref_key": "language", "pref_value": {"locale": "en-US"}, "source": "profile",
  fact       score=0.931  {"fact_id": "fact_b4fc6396de81", "content": "Stripe webhook URL is https://api.a
  fact       score=0.786  {"fact_id":

## 9. Retrieval cascade fallback

`assemble()` walks `hybrid -> vector -> lexical -> like`. The caller
asks for a starting tier; on Oracle errors (missing Text index,
unregistered embedding model) the function degrades. `RetrievalResult.mode`
reflects the tier that actually served.


In [11]:
from memory.retrieval import assemble

async def show_tiers():
    aconn = await connect()
    try:
        for tier in ("hybrid", "vector", "lexical", "like"):
            r = await assemble(
                aconn, TENANT, USER, "run_demo_nb",
                "stripe webhook", mode=tier,
            )
            print(f"  requested={tier:<8} served={r.mode:<8} "
                  f"counts={r.counts()}")
    finally:
        await aconn.close()

await show_tiers()


  requested=hybrid   served=hybrid   counts={'policy': 5, 'preference': 3, 'fact': 5, 'episodic': 3}
  requested=vector   served=vector   counts={'policy': 5, 'preference': 3, 'fact': 5, 'episodic': 3}


  requested=lexical  served=lexical  counts={'policy': 5, 'preference': 3, 'fact': 1}
  requested=like     served=like     counts={'policy': 5, 'preference': 3, 'fact': 3, 'episodic': 1}


## 10. Episode retrieval — lexical catches verbatim tokens

Episodic memory now runs the same hybrid pattern as facts: the
`idx_ep_text` CONTEXT index on `summary` lets the lexical CTE catch
ticket IDs, error codes, and other distinctive tokens that pure vector
similarity tends to dilute. We seed a one-off episode whose summary
names `STRIPE-1247`, then search for that exact token.


In [12]:
from memory.stores.episodic import EpisodicStore

async def episode_lexical_demo():
    aconn = await connect()
    try:
        await EpisodicStore(aconn).write(
            tenant_id=TENANT, task_type="support_case",
            title="Escalation on STRIPE-1247 webhook signature",
            summary=("Customer rotated webhook secret; ticket STRIPE-1247 "
                     "routed to tier 2 and resolved by env-var redeploy."),
            outcome="resolved",
            key_steps=["compared dashboard secret to env var",
                       "redeployed", "verified delivery"],
            source_run_id="run_demo_nb",
        )
        result = await assemble(
            aconn, TENANT, USER, "run_demo_nb", "STRIPE-1247",
        )
        for row in result.by_kind("episodic"):
            score = f"{row.rank_score:.3f}" if row.rank_score else "  —  "
            print(f"  score={score}  {row.payload['title']}")
    finally:
        await aconn.close()

await episode_lexical_demo()


  score=0.659  Escalation on STRIPE-1247 webhook signature
  score=0.550  Refund failed due to currency-locale mismatch
  score=0.515  Stripe webhook signature mismatch after secret rotation


## 11. The promotion gate

Three hand-constructed candidates illustrate the three paths through the gate:
dedup hit, low-confidence reject, contradiction → supersession.

In [13]:
from memory.extraction import RuleBasedExtractor
from memory.manager import MemoryManager
from memory.model import SimulatedModel
from memory.records import MemoryCandidate

def show(label, r):
    """PromotionResult carries six fields: outcome (gate verdict),
    record_id, reason, status (resulting record's lifecycle), and
    confidence (the candidate's source confidence echoed back)."""
    print(f"{label:<20} outcome={r.outcome:<13} status={str(r.status):<12} "
          f"conf={r.confidence}  record_id={r.record_id}  reason={r.reason}")

async def gate_demo():
    aconn = await connect()
    manager = MemoryManager(aconn, SimulatedModel(), RuleBasedExtractor())

    # Case A: dedup hit. The seed already has a v1 webhook URL fact.
    dup = MemoryCandidate(
        memory_type="fact", tenant_id=TENANT, user_id=USER,
        content="Stripe webhook URL is https://api.acme.com/v1/stripe",
        confidence=0.95, source_run_id="run_demo",
        subject=f"customer:{USER}", predicate="infrastructure",
    )
    show("Case A (dedup):", await manager.gate.promote(dup))

    # Case B: low confidence — rejected by the 0.7 threshold.
    low = MemoryCandidate(
        memory_type="fact", tenant_id=TENANT, user_id=USER,
        content="Customer might be migrating to GCP", confidence=0.4,
        source_run_id="run_demo",
        subject=f"customer:{USER}", predicate="speculation",
    )
    show("Case B (low-conf):", await manager.gate.promote(low))

    # Case C: contradiction → supersession. The new fact lands as 'active'
    # because the contradiction itself is the confirmation event.
    contra = MemoryCandidate(
        memory_type="fact", tenant_id=TENANT, user_id=USER,
        content="Stripe webhook URL is https://api.acme.com/v3/stripe",
        confidence=0.95, source_run_id="run_demo",
        subject=f"customer:{USER}", predicate="infrastructure",
    )
    show("Case C (supersede):", await manager.gate.promote(contra))

    await aconn.close()

await gate_demo()


Case A (dedup):      outcome=deduplicated  status=None         conf=0.95  record_id=fact_b4fc6396de81  reason=None
Case B (low-conf):   outcome=rejected      status=None         conf=0.4  record_id=None  reason=low_confidence


Case C (supersede):  outcome=superseded    status=active       conf=0.95  record_id=fact_9faa6c028941  reason=superseded:fact_b4fc6396de81


## 12. Turn loop — Run 1

Four scripted turns demonstrating the full loop. Reset the tenant first
so the scenario starts from a known state.

In [14]:
# Reset and re-seed.
from memory.ddl import drop_all, create_all
import data.seed
drop_all(conn); create_all(conn)
await data.seed.seed()

from memory.agent_session import AgentSession

async def run1():
    aconn = await connect()
    manager = MemoryManager(aconn, SimulatedModel(), RuleBasedExtractor())
    session = AgentSession(
        tenant_id=TENANT, user_id=USER,
        agent_id=os.getenv("DEMO_AGENT_ID", "agent:support_v1"),
    )

    turns = [
        "Hi, I'm having trouble with my Stripe webhook again.",
        "By the way, I prefer terse answers — skip the pleasantries.",
        "Also, our production webhook URL is now https://api.acme.com/v2/stripe",
        "Thanks, that's all for now.",
    ]
    for i, msg in enumerate(turns, 1):
        print(f"\n--- Turn {i}: {msg!r}")
        response, context, promotions = await manager.handle_turn(session, msg)
        print(f"  retrieved: {len(context.facts)} facts, {len(context.preferences)} prefs, "
              f"{len(context.episodes)} episodes, {len(context.recent)} recent")
        print(f"  model    : {response.text}")
        for p in promotions:
            print(f"  promote  : {p.outcome} — {p.record_id or p.reason}")

    await aconn.close()
    return session

run1_session = await run1()

Seeded tenant acme-support: 5 policies, 3 preferences, 8 facts, 3 episodes.

--- Turn 1: "Hi, I'm having trouble with my Stripe webhook again."
  retrieved: 5 facts, 3 prefs, 3 episodes, 1 recent
  model    : Got it. I see you said: "Hi, I'm having trouble with my Stripe webhook again.". Drawing on what we already know, here's an acknowledgement.

--- Turn 2: 'By the way, I prefer terse answers — skip the pleasantries.'


  retrieved: 5 facts, 3 prefs, 3 episodes, 4 recent
  model    : Acknowledged: By the way, I prefer terse answers — skip the pleasantries.
  promote  : written — verbosity

--- Turn 3: 'Also, our production webhook URL is now https://api.acme.com/v2/stripe'
  retrieved: 5 facts, 4 prefs, 3 episodes, 5 recent
  model    : Acknowledged: Also, our production webhook URL is now https://api.acme.com
  promote  : superseded — fact_59fd41ed99d6

--- Turn 4: "Thanks, that's all for now."
  retrieved: 5 facts, 4 prefs, 3 episodes, 5 recent
  model    : Acknowledged: Thanks, that's all for now.


## 13. The turn envelope

Every turn ends with a `turn_envelope` trace event. Each promotion
entry carries six fields — `type`, `fact_id`, `confidence`, `status`,
`outcome`, `reason` — so replay can reconstruct both the gate verdict
and the resulting record state without re-running extraction.


In [15]:
from memory.stores.trace import TraceStore

async def inspect_envelope():
    aconn = await connect()
    try:
        events = await TraceStore(aconn).get_run(run1_session.run_id)
        envs = [e for e in events if e["event_type"] == "turn_envelope"]
        print(f"  envelopes recorded: {len(envs)}")
        for env in envs:
            promos = env["payload"].get("promotions", [])
            if not promos:
                continue
            print(f"\n  turn {env['turn_index']} promotions:")
            for p in promos:
                print(f"    {p}")
    finally:
        await aconn.close()

await inspect_envelope()


  envelopes recorded: 4

  turn 1 promotions:
    {'type': 'preference', 'fact_id': None, 'confidence': Decimal('1'), 'status': None, 'outcome': 'written', 'reason': None}

  turn 2 promotions:
    {'type': 'fact', 'fact_id': 'fact_59fd41ed99d6', 'confidence': Decimal('0.95'), 'status': 'active', 'outcome': 'superseded', 'reason': 'superseded:fact_999a49f0f739'}


## 14. Run 2 — resumability

A new `AgentSession` is constructed (simulating process restart on Monday
after closing the tab on Friday). Same identity, same durable layer.
The verbosity preference and the new webhook URL are still here.

Before the new turn, we confirm the provisional v2 fact via the admin path —
this is what a real verification gate would do.

In [16]:
async def run2(prev_session):
    aconn = await connect()
    manager = MemoryManager(aconn, SimulatedModel(), RuleBasedExtractor())

    # Find the provisional v2 fact and confirm it.
    cur2 = aconn.cursor()
    await cur2.execute(
        "SELECT fact_id FROM fact_memory WHERE tenant_id = :tid AND status = 'provisional' "
        "AND superseded_by IS NULL ORDER BY created_at DESC FETCH FIRST 1 ROWS ONLY",
        tid=TENANT,
    )
    row = await cur2.fetchone()
    if row:
        await manager.fact.confirm(row[0])
        print(f"  confirmed provisional fact {row[0]}")

    # Fresh AgentSession — same identity, new run.
    session = AgentSession(
        tenant_id=prev_session.tenant_id,
        user_id=prev_session.user_id,
        agent_id=prev_session.agent_id,
    )
    print(f"  new run: {session.run_id}\n")

    response, context, _ = await manager.handle_turn(
        session, "Hey, what did we change yesterday about the webhook?"
    )
    print("preferences carried over:", [p["pref_key"] for p in context.preferences])
    print("facts retrieved:", [f["content"] for f in context.facts])
    print("\nmodel:", response.text)

    await aconn.close()

await run2(run1_session)

  new run: run_0846f710



preferences carried over: ['timezone', 'notification_channel', 'verbosity', 'language']
facts retrieved: ['Webhook retries follow exponential backoff: 1s, 4s, 16s, 64s, then daily for 3 days before the event is dropped.', 'Stripe webhook URL is https://api.acme.com/v2/stripe', 'Stripe webhooks are signed with HMAC-SHA256 over the raw request body; the signature is sent in the Stripe-Signature header.', 'API rate limits: 100 read requests per second and 25 write requests per second per account in live mode.', 'Account tier: Enterprise (signed 2024-Q3, renews 2026-Q3)']

model: Acknowledged: Hey, what did we change yesterday about the webhook?


## 15. GDPR cascade & cleanup

Right-to-forget: revoke facts and episodes for the user, delete preferences and
trace, write the audit row. All in one transaction.

In [17]:
async def erase():
    aconn = await connect()
    manager = MemoryManager(aconn, SimulatedModel(), RuleBasedExtractor())
    result = await manager.cascade_erase(USER, TENANT)
    print(f"erased user: {result}")
    cur2 = aconn.cursor()
    await cur2.execute(
        "SELECT scope, deleted_at, reason FROM deletion_events WHERE user_id = :u_id",
        u_id=USER,
    )
    async for row in cur2:
        print(f"  audit: {row}")
    await aconn.close()

await erase()


erased user: {'user_id': 'jane_doe@example.com', 'tenant_id': 'acme-support', 'reason': 'gdpr_erasure'}
  audit: ('all', datetime.datetime(2026, 5, 14, 18, 26, 19), 'gdpr_erasure')


## 16. Optional: drop the demo tables

Run this only when you want a clean schema. The CLI app and the
rest of this notebook will fail with `ORA-00942` until you re-run
`python -m memory.ddl setup && python -m data.seed` (or restart
this notebook from Section 2). Skip this cell to leave the seeded
tenant in place for further exploration.


In [18]:
drop_all(conn)
conn.close()
print("Tables dropped. Re-run Section 2 (and Section 4 to re-seed)"
      " before using the CLI or this notebook again.")


Tables dropped. Re-run Section 2 (and Section 4 to re-seed) before using the CLI or this notebook again.
